<a href="https://colab.research.google.com/github/maciekpopik/ENEN-645-Group-4-Final/blob/main/PlantLab2RealGeneralization_CutMix%2BGradCAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
#Imports

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2

In [35]:
#Parameters

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
FEWSHOT_EPOCHS = 5
LR = 3e-4
NUM_WORKERS = 2



In [36]:
#Data transforms (data augmentation + normalization)

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(
        IMG_SIZE, scale=(0.5, 1.0), ratio=(0.75, 1.33)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.3, contrast=0.35,
        saturation=0.4, hue=0.08
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [37]:
#Dataset Paths & Loaders - PlantVillage and PlantDoc dataset

ROOT = "/content/drive/MyDrive/PlantLab2RealGeneralization"

pv_train = datasets.ImageFolder(f"{ROOT}/Train", train_tfms)
pv_val   = datasets.ImageFolder(f"{ROOT}/Val", eval_tfms)
pv_test  = datasets.ImageFolder(f"{ROOT}/Test_ID", eval_tfms)

pd_few   = datasets.ImageFolder(f"{ROOT}/Few_Shot", train_tfms)
pd_test  = datasets.ImageFolder(f"{ROOT}/Test_OOD", eval_tfms)

train_loader = DataLoader(pv_train, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(pv_val, BATCH_SIZE, shuffle=False)
pv_test_loader = DataLoader(pv_test, BATCH_SIZE, shuffle=False)
pd_few_loader  = DataLoader(pd_few, BATCH_SIZE, shuffle=True)
pd_test_loader = DataLoader(pd_test, BATCH_SIZE, shuffle=False)

NUM_CLASSES = len(pv_train.classes)
class_names = pv_train.classes

In [39]:
#Model using EfficientNet

model = models.efficientnet_b0(weights="IMAGENET1K_V1")

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

model = model.to(DEVICE)

In [40]:
#CutMix

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size, _, H, W = x.size()
    index = torch.randperm(batch_size).to(x.device)

    cut_ratio = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_ratio)
    cut_h = int(H * cut_ratio)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))

    return x, y, y[index], lam

In [41]:
def cutmix_loss(pred,y1,y2,lam):
    return lam*nn.CrossEntropyLoss()(pred,y1) + (1-lam)*nn.CrossEntropyLoss()(pred,y2)

In [42]:
#Training and Evaluation

def train_epoch(loader, cutmix=True):
    model.train()
    for x,y in tqdm(loader):
        x,y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        if cutmix and np.random.rand() < 0.5:
            x,y1,y2,lam = cutmix_data(x,y)
            out = model(x)
            loss = cutmix_loss(out,y1,y2,lam)
        else:
            loss = nn.CrossEntropyLoss()(model(x),y)
        loss.backward()
        optimizer.step()

In [43]:
def evaluate(loader):
    model.eval()
    correct,total = 0,0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(1)
            correct += (pred==y).sum().item()
            total += y.size(0)
    return correct/total

In [44]:
#Training on PlantVillage Dataset

optimizer = optim.AdamW(model.parameters(), lr=LR)

for e in range(EPOCHS):
    train_epoch(train_loader, cutmix=True)
    val_acc = evaluate(val_loader)
    print(f"Epoch {e+1}: Val Acc {val_acc:.3f}")

  0%|          | 0/866 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [45]:
#Test on PlantVillage Test Dataset (Test_ID)

pv_test_acc = evaluate(pv_test_loader)
print(f"PlantVillage Test_ID Accuracy: {pv_test_acc*100:.2f}%")

KeyboardInterrupt: 

In [46]:
#PlantDoc FewShot Dataset

for param in model.features.parameters():
    param.requires_grad = False

optimizer = optim.Adam(model.classifier.parameters(), lr=1e-4)

for e in range(FEWSHOT_EPOCHS):
    train_epoch(pd_few_loader, cutmix=False)

  0%|          | 0/10 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [47]:
#Test on PlantDoc Dataset (Test_OOD)

pd_acc = evaluate(pd_test_loader)
print(f"PlantDoc Accuracy (after few‑shot): {pd_acc*100:.2f}%")

KeyboardInterrupt: 

In [48]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, inp, out):
        self.activations = out

    def save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def generate(self, class_idx):
        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)
        cam = F.relu(cam)
        cam -= cam.min()
        cam /= cam.max() + 1e-8
        return cam

In [49]:
target_layer = model.features[-1]
gradcam = GradCAM(model, target_layer)

In [52]:
#GradCAM on PlantVillage dataset (Test_ID)

images, labels = next(iter(pv_test_loader))
image = images[0].unsqueeze(0).to(DEVICE)
image.requires_grad_(True)

model.eval()

image = images[0].unsqueeze(0).to(DEVICE)
image.requires_grad_(True)

output = model(image)
pred = output.argmax(dim=1).item()

model.zero_grad()
output[0, pred].backward()

cam = gradcam.generate(pred)[0].detach().cpu().numpy()

In [53]:
#GradCAM on PlantDoc dataset  (Test_OOD)

images, labels = next(iter(pd_test_loader))
image = images[0].unsqueeze(0).to(DEVICE)
image.requires_grad_(True)

model.eval()

image = images[0].unsqueeze(0).to(DEVICE)
image.requires_grad_(True)

output = model(image)
pred = output.argmax(dim=1).item()

model.zero_grad()
output[0, pred].backward()

cam = gradcam.generate(pred)[0].detach().cpu().numpy()